In [ ]:
import pandas as pd

In [2]:
import numpy as np

# Read and merge dicom download files

Load the CSV file(s) from the DICOM download from LONI IDA. This is the image collection CSV that you created on LONI IDA. If you created multiple collections in LONI and downloaded them independently, they must be merged as shown in the example below. 

In [ ]:
adni_all = pd.read_csv('/path/to/loni_download.csv')

In [ ]:
# If needed, merge all files into one big dataframe.
## If you have multiple csv files to merge, merge them two at a time as shown below.
# adni_all = pd.merge(adni_mri, adni_pet, how='outer')
# adni_all = pd.merge(adni_all, adni_dti, how='outer')

In [ ]:
## Each row corresponds to a directory with DICOMs associated to a single Series with it's Image ID. 
#### The download sheet does NOT indicate how many dcm files are contained in each directory. 
print('This is the shape of the data-frame: ',adni_all.shape)
print(adni_all.columns)

In [6]:
# might need to fix merge error by changing visit code to object dtype.
adni_all['Visit'] = adni_all['Visit'].astype('object')

In [ ]:
# Check the number of unique subjects. 
print('Unique Subject IDs: ',len(adni_all['Subject'].value_counts()))
# Check the number of unique Modalities. 
print('Modality types: ',adni_all['Modality'].value_counts())

In [ ]:
# Check other fields
print('Other types: ',adni_all['Type'].value_counts())

In [ ]:
# Chech the number of unique directory names for the DICOM directories.
unique_series = adni_all['Description'].value_counts()
print('Total Number of Unique Series Descriptions is ', len(unique_series))
print(unique_series[0:50])


In [10]:
adni_all[adni_all['Description'].str.contains('Sag IR-SPGR',case=False, na=False)];

# Building the resting state fMRI collection

In [11]:
# This is Saige's diectionary. Not exactly sure how she built it but I built my own basedon string matches
#  and I ended up with the a handful more series than if I had used her dictionary. 
# Build a dictionary of all possible fMRI DICOM directory names.
fmri_names_saige = ['Axial rsfMRI (Eyes Open)',
'Resting State fMRI',
'Axial MB rsfMRI (Eyes Open)',
'Axial fcMRI (EYES OPEN)',
'Axial rsfMRI (EYES OPEN)',
'Axial fcMRI (Eyes Open)',
'Extended Resting State fMRI',
'Axial rsfMRI (Eyes Open) -phase P to A',
'Axial fcMRI 0 angle (EYES OPEN)',
'Axial fcMRI',
'Axial MB rsfMRI (Eyes Open)   straight no angle',
'Axial RESTING fcMRI (EYES OPEN)',
'Axial MB rsfMRI AP',
'Axial rsfMRI (Eyes Open) 10 min :-PJ',
'Extended AXIAL rsfMRI EYES OPEN',
'Axial rsfMRI (Eyes Open) Phase Direction P>A',
'AXIAL RS fMRI (EYES OPEN)',
'Axial_rsFMRI_Eyes_Open',
'Extended Resting State fMRI CLEAR',
'Axial MB rsfMRI (Eyes Open) REPEAT',
'Axial fcMRI (EYES OPEN)_REPEAT',
'Axial rsfMRI P>>A (Eyes Open)',
'Axial MB rsfMRI',
'Axial MB rsfMRI (Eyes Open) (MSV21)',
'epi_2s_resting_state',
'Axial rsfMRI (Eyes Open) 10 min']

In [ ]:
# I built my rsfMRI collection by finding all the series that include fMRI, fcMRI and epi in them. 
# I tried other substrings but these are the ones that I think cover the entire set
fMRIstr = adni_all[adni_all['Description'].str.contains('fMRI',case=False, na=False)]
fcMRIstr = adni_all[adni_all['Description'].str.contains('fcMRI',case=False, na=False)]
epistr = adni_all[adni_all['Description'].str.contains('epi',case=False, na=False)]
print('====== Matching fMRI string: ',len(fMRIstr))
print('======== Unique Series Descriptions: ',len(fMRIstr['Description'].unique()))
print(fMRIstr['Description'].value_counts())
print('====== Matching fcMRI string: ',len(fcMRIstr))
print('======== Unique Series Descriptions: ',len(fcMRIstr['Description'].unique()))
print(fcMRIstr['Description'].value_counts())
print('====== Matching epi string: ',len(epistr))
print(epistr['Description'].value_counts())
print('======== Unique Series Descriptions: ',len(epistr['Description'].unique()))
#print(fMRIstr)

In [ ]:
# Concatenate the different matches
fmri_series_desc_names = np.concatenate([fMRIstr['Description'].unique(),fcMRIstr['Description'].unique(),epistr['Description'].unique()])
# remove potential repetitions
fmri_series_desc_names = np.unique(fmri_series_desc_names)
fmri_series_dirs = adni_all[adni_all['Description'].isin(fmri_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories found in collection: ',fmri_series_dirs.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of Subjects with fMRI data: ',len(fmri_series_dirs['Subject'].value_counts()))

In [ ]:
# TO KEEP IN MIND: Don't rely on Modality tags to sort data - there is rsfMRI with non fMRI Modalities. 
fmri_series_dirs_test = adni_all[adni_all['Description'].isin(fmri_series_desc_names) & (adni_all['Modality'] != 'fMRI')]
print(fmri_series_dirs_test.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print(len(fmri_series_dirs_test['Subject'].value_counts()))
print(fmri_series_dirs_test['Description'].value_counts())

In [ ]:
## Cross-checking with Saige's list of fMRI series descriptions. 
fmri_series_dirs_saige = adni_all[adni_all['Description'].isin(fmri_names_saige)]
# Check the number of fMRI DICOM directories.
print('shape of fmri_series_dirs_saige: ',fmri_series_dirs_saige.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of unique subject IDs: ',len(fmri_series_dirs_saige['Subject'].value_counts()))

Pretty similar - Not sure where the discrepancy is coming from but it is just 3 folders that don't match

# Building the MPRAGE collection

In [ ]:
test_mprage = adni_all[adni_all['Description'].str.contains('_nd',case=False, na=False)]
print('====== Matching mprage string: ',len(test_mprage))
print('======== Unique Series Descriptions: ',len(test_mprage['Description'].unique()))
test_mprage['Description'].unique()

In [ ]:
# I built the MPRAGE collection by finding all the series that include MPRAGE,MP-RAGE,SPGR and T1 
# I tried other substrings but these are the ones that I think cover the entire set
mprage = adni_all[adni_all['Description'].str.contains('MPRAGE',case=False, na=False)]
mp_rage = adni_all[adni_all['Description'].str.contains('MP-RAGE',case=False, na=False)]
SPGR = adni_all[adni_all['Description'].str.contains('SPGR',case=False, na=False)]
t1s = adni_all[adni_all['Description'].str.contains('T1',case=False, na=False)]
print('====== Matching mprage string: ',len(mprage))
print('======== Unique Series Descriptions: ',len(mprage['Description'].unique()))
#print(mprage['Description'].value_counts())
print('====== Matching mp-rage string: ',len(mp_rage))
print('======== Unique Series Descriptions: ',len(mp_rage['Description'].unique()))
#print(mp_rage['Description'].value_counts())
print('====== Matching SPGR string: ',len(SPGR))
print('======== Unique Series Descriptions: ',len(SPGR['Description'].unique()))
#print(SPGR['Description'].value_counts())
print('====== Matching T1 string: ',len(t1s))
print('======== Unique Series Descriptions: ',len(t1s['Description'].unique()))
#print(t1s['Description'].value_counts())
#print(fMRIstr)

In [ ]:
# Concatenate the different matches
mprage_series_desc_names = np.concatenate([mprage['Description'].unique(),mp_rage['Description'].unique(),SPGR['Description'].unique(),t1s['Description'].unique()])
#print(len(mprage_series_desc_names))
mprage_series_desc_names = np.unique(mprage_series_desc_names)
print('Unique series descriptions for MPRAGE: ',len(mprage_series_desc_names))

mprage_series_dirs = adni_all[adni_all['Description'].isin(mprage_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories found in collection: ',mprage_series_dirs.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of Subjects with T1 data: ',len(mprage_series_dirs['Subject'].value_counts()))

In [ ]:
mprage_series_dirs['Description'].value_counts()

## Read the csv listing all the files extracted from the zip downloads from LONI

Before loading the file that lists all the unzipped DICOM data, I had to do some editing of the Series Descriptions. 
Specifically, the CSV provided by LONI has removed underscores and parenthesis from the description names. 

First I made a copy of the CSV file so that I could preserve the original file names as they appear in my unzipped DICOM data. I added `_edited.csv` to the copy to distinguish the files. 

Then I opened up the edited CSV file with VIM and did the following substitutions in this order:
 - `:%s/_EYES_OPEN_/(EYES OPEN)/g`  => adds parenthesys around the `open eyes` strings
 - `:%s/_Eyes_Open_/(Eyes Open)/g`
 - `:%s/_/ /g`                      => substitutes all underscores for spaces
 - `:%s/  -PJ/ :-PJ/g`               => adds colon to the PJ strings
 - `:%s/Direction P A/Direction P>A/g`  => adds the > for the P=>A direction 
 - `:%s/w acceleration/w\/acceleration/g`
 - `:%s/MPRAGE S2 DIS3D/MPRAGE_S2_DIS3D/g`
 - `:%s/MPRAGE ASO repeat/MPRAGE_ASO_repeat/n`
 
 
 
 This worked for resting state fMRI. I haven't yet checked other series.  

In [ ]:
# Load the file created from just listing the unzipped DICOM directories.
# I did this in an attempt to verify the download. 
# This list should match the all_data list if the download was successful.
adni_unzip_path="/N/project/cfn-rady/andrea-dev/ADNI/LONI_data/image_data/ADNI/unzipped_dicom_dirs_inventory_edited.csv"
adni_unzip=pd.read_csv(adni_unzip_path)
print(adni_unzip.columns)
print('Number of Series Directories unzipped: ',adni_unzip.shape)
# Check the number of unique subjects. 
print('Unique Subject IDs: ',len(adni_unzip['Subject'].value_counts()))

Test if the edits I made to series descriptin names align the with the descriptions from the LONI downloaded metadata

In [ ]:
## fMRI
fmri_series_dirs_unzip = adni_unzip[adni_unzip['Description'].isin(fmri_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories unzipped: ',fmri_series_dirs_unzip.shape)
print('Number of unzipped Subjects with fMRI data: ',len(fmri_series_dirs_unzip['Subject'].value_counts()))

print('Number of directories listed in metadata: ',fmri_series_dirs.shape)
print('Number of Subjects listed in metadata: ',len(fmri_series_dirs['Subject'].value_counts()))

#print([fmri_series_dirs_unzip['Description'].value_counts(),fmri_series_dirs['Description'].value_counts()])

In [ ]:
## Test if the edits I made to series descriptin names align the with the descriptions from the LONI downloaded metadata
fMRIstr_zip = adni_unzip[adni_unzip['Description'].str.contains('fMRI',case=False, na=False)]
fcMRIstr_zip = adni_unzip[adni_unzip['Description'].str.contains('fcMRI',case=False, na=False)]
epistr_zip = adni_unzip[adni_unzip['Description'].str.contains('epi',case=False, na=False)]
print('====== Matching fMRI string: ',len(fMRIstr_zip))
print('======== Unique Series Descriptions: ',len(fMRIstr_zip['Description'].unique()))
#print(fMRIstr_zip['Description'].value_counts())
print('====== Matching fcMRI string: ',len(fcMRIstr_zip))
print('======== Unique Series Descriptions: ',len(fcMRIstr_zip['Description'].unique()))
#print(fcMRIstr_zip['Description'].value_counts())
print('====== Matching epi string: ',len(epistr_zip))
print(epistr_zip['Description'].value_counts())
#print('======== Unique Series Descriptions: ',len(epistr_zip['Description'].unique()))
#print(fMRIstr)

In [ ]:
## MPRAGE
mprage_series_dirs_unzip = adni_unzip[adni_unzip['Description'].isin(mprage_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories unzipped: ',mprage_series_dirs_unzip.shape)
print('Number of unzipped Subjects with MPRAGE data: ',len(mprage_series_dirs_unzip['Subject'].value_counts()))

print('Number of directories listed in metadata: ',mprage_series_dirs.shape)
print('Number of Subjects listed in metadata: ',len(mprage_series_dirs['Subject'].value_counts()))

#print([mprage_series_dirs_unzip['Description'].value_counts()[0:50],mprage_series_dirs['Description'].value_counts()[0:50]])

In [ ]:
# This cell was used to list out the DICOM directory names.
# Then I had to manually go through and chose the names that are for fMRI scans, which I input to the dictionary in the cell below.
#pd.set_option('display.max_rows', None)

In [ ]:
# Build a dictionary of all the fMRI DICOM directory names
fmri_dcm = ['Axial_rsfMRI__Eyes_Open_','Resting_State_fMRI','Axial_MB_rsfMRI__Eyes_Open_', 'Axial_fcMRI__EYES_OPEN_','Axial_rsfMRI__EYES_OPEN_',
'Extended_Resting_State_fMRI','Axial_fcMRI__Eyes_Open_', 'Axial_rsfMRI__Eyes_Open__-phase_P_to_A', 'Axial_RESTING_fcMRI__EYES_OPEN_',
'Axial_rsfMRI__Eyes_Open__10_min__-PJ','Axial_MB_rsfMRI__Eyes_Open____straight_no_angle','Axial_rsfMRI__Eyes_Open__Phase_Direction_P_A',
'Extended_AXIAL_rsfMRI_EYES_OPEN','AXIAL_RS_fMRI__EYES_OPEN_', 'Axial_rsFMRI_Eyes_Open','Axial_MB_rsfMRI__Eyes_Open__REPEAT',
'Axial_rsfMRI__Eyes_Open__10_min','epi_2s_resting_state','Axial_fcMRI__EYES_OPEN__REPEAT','Axial_MB_rsfMRI__Eyes_Open___MSV21_',
'Axial_MB_rsfMRI','Axial_-_Advanced_fMRI_64_Channel','Axial_rsfMRI_P__A__Eyes_Open_']

In [ ]:
# Build a new dataframe that only contains the fMRI DICOM directories.
fmri_dcm_df = dcm_df[dcm_df['scan_name'].isin(fmri_dcm)]

In [ ]:
# Check the number of fMRI DICOM directories.
fmri_dcm_df.shape

In [ ]:
# Check the number of unique subjects with fMRI DICOM directory.
len(fmri_dcm_df['Subject_ID'].value_counts())